<a href="https://colab.research.google.com/github/Nurfinkaa/MachineLearning/blob/main/js6/TG6_6_ANN_244107020211_NURFINKA_LAILASARI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Praktikum 6**

In [2]:
!pip install faiss-cpu annoy hnswlib

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 647.5/647.5 kB 5.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 66.6 MB/s eta 0:00:00
  Created wheel for annoy: filename=annoy-1.17.3-cp313-cp313-linux_x86_64.whl size=520035 sha256=6cb3f7a199044346b44e339d429fa3545ed625cdafc2a9a2eb8f45e6458418ce
  Stored in directory: /root/.cache/pip/wheels/fc/4e/87/f1f957e7382aa370452cff98276f51686924d5415e449800ed
  Created wheel for hnswlib: filename=hnswlib-0.8.0-cp313-cp313-linux_x86_64.whl size=2679795 sha256=ce8bbbb9fd026e678cea8ce90db4b82e419793708b1320453b5849fefbef59d7
  Stored in directory: /root/.cache/pip/wheels/35/04/88/b31765a4b9957705e18065db4657e61fc8da54f50e3ef0b67e
Successfully built annoy hnswlib


In [10]:
import glob
import os
import time
import numpy as np
import pandas as pd
import faiss
import hnswlib
import kagglehub
from annoy import AnnoyIndex
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

**Unduh dataset dan cari file yang berisi fitur audio**

In [19]:
path = kagglehub.dataset_download("bwandowando/spotify-songs-with-attributes-and-lyrics")
print("Path to dataset files:", path)

features = ['danceability', 'energy', 'loudness', 'speechiness',
            'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo']

csv_files = glob.glob(os.path.join(path, "**", "*.csv"), recursive=True)
print("\nSemua file CSV dan kolomnya:")
cocok = []
for f in csv_files:
    cols = pd.read_csv(f, nrows=0).columns.tolist()
    ada = all(c in cols for c in features)
    print(f"- {os.path.basename(f)} ({os.path.getsize(f)/1e6:.1f} MB) | semua fitur ada: {ada}")
    if not ada:
        print("   kolom:", cols[:12], "..." if len(cols) > 12 else "")
    else:
        cocok.append(f)

if not cocok:
    raise ValueError("Tidak ada CSV yang berisi semua kolom fitur. Kirim daftar kolom di atas ke Claude.")

csv_path = cocok[0]
print("\nDipakai:", csv_path)

Using Colab cache for faster access to the 'spotify-songs-with-attributes-and-lyrics' dataset.
Path to dataset files: /kaggle/input/spotify-songs-with-attributes-and-lyrics

Semua file CSV dan kolomnya:
- songs_with_lyrics_and_timestamps.csv (2420.5 MB) | semua fitur ada: False
   kolom: ['id', 'startTimeMs', 'words'] 
- songs_with_attributes_and_lyrics.csv (1545.1 MB) | semua fitur ada: True

Dipakai: /kaggle/input/spotify-songs-with-attributes-and-lyrics/songs_with_attributes_and_lyrics.csv


**Baca data**

In [12]:
df = pd.read_csv(csv_path, usecols=features).dropna()   # hanya kolom fitur, buang nilai kosong
print("Jumlah lagu:", len(df))

# Opsional: kalau datanya terlalu besar/lambat, hilangkan tanda # di baris berikut
# df = df.sample(200_000, random_state=42)

df.head()

Jumlah lagu: 955320


,danceability,energy,loudness,speechiness,acousticness,instrumentalness,liveness,valence,tempo
0,0.415,0.6050,-11.157,0.0575,0.00116,0.838000,0.4710,0.193,100.059
1,0.788,0.6480,-9.135,0.3150,0.90000,0.000000,0.1760,0.287,79.998
2,0.000,0.0354,-20.151,0.0000,0.90800,0.000000,0.4790,0.000,0.000
3,0.657,0.8820,-6.340,0.0385,0.00740,0.000013,0.0474,0.939,123.588
4,0.659,0.8930,-8.531,0.0411,0.09220,0.000019,0.0534,0.951,123.600


In [14]:
X = df[features].values

# Standarisasi fitur (float32 supaya kompatibel dengan FAISS dan hnswlib)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X).astype('float32')
n, dim = X_scaled.shape

In [15]:
# -------------------------------
# Query: 1000 lagu acak
# -------------------------------
k = 10
n_query = min(1000, n)
rng = np.random.default_rng(42)
q_idx = rng.choice(n, size=n_query, replace=False)
Xq = X_scaled[q_idx]

faiss.omp_set_num_threads(1)   # 1 thread supaya perbandingan waktu adil

def recall_at_k(pred, truth, k):
    return sum(len(set(p[:k]) & set(t[:k])) for p, t in zip(pred, truth)) / (len(truth) * k)

hasil = []

In [16]:
# -------------------------------
# Exact Nearest Neighbor (brute-force)
# -------------------------------
start = time.perf_counter()
nn = NearestNeighbors(n_neighbors=k, algorithm='brute', metric='euclidean')
nn.fit(X_scaled)
t_build = time.perf_counter() - start

start = time.perf_counter()
dist_exact, idx_exact = nn.kneighbors(Xq)
t_query = time.perf_counter() - start
hasil.append(["Exact NN (brute)", t_build, t_query / n_query * 1000, 1.0])
print(f"Exact NN selesai: query {t_query:.3f} s")

Exact NN selesai: query 5.334 s


In [17]:
# -------------------------------
# Annoy
# -------------------------------
start = time.perf_counter()
index_annoy = AnnoyIndex(dim, 'euclidean')
index_annoy.set_seed(42)
for i, v in enumerate(X_scaled):
    index_annoy.add_item(i, v)
index_annoy.build(10)
t_build = time.perf_counter() - start

start = time.perf_counter()
idx_annoy = [index_annoy.get_nns_by_vector(v, k) for v in Xq]
t_query = time.perf_counter() - start
hasil.append(["Annoy (10 tree)", t_build, t_query / n_query * 1000, recall_at_k(idx_annoy, idx_exact, k)])
print(f"Annoy selesai: build {t_build:.3f} s, query {t_query:.3f} s")

Annoy selesai: build 20.703 s, query 0.053 s


In [18]:
# -------------------------------
# HNSW
# -------------------------------
start = time.perf_counter()
p_hnsw = hnswlib.Index(space='l2', dim=dim)
p_hnsw.init_index(max_elements=n, ef_construction=200, M=16, random_seed=42)
p_hnsw.add_items(X_scaled)
p_hnsw.set_ef(200)
t_build = time.perf_counter() - start

start = time.perf_counter()
idx_hnsw = [p_hnsw.knn_query(v.reshape(1, -1), k=k)[0][0] for v in Xq]
t_query = time.perf_counter() - start
hasil.append(["HNSW (M=16, ef=200)", t_build, t_query / n_query * 1000, recall_at_k(idx_hnsw, idx_exact, k)])
print(f"HNSW selesai: build {t_build:.3f} s, query {t_query:.3f} s")

HNSW selesai: build 226.082 s, query 0.250 s


In [20]:
# -------------------------------
# FAISS IVF
# -------------------------------
start = time.perf_counter()
quantizer = faiss.IndexFlatL2(dim)
index_faiss = faiss.IndexIVFFlat(quantizer, dim, 100, faiss.METRIC_L2)
index_faiss.train(X_scaled)
index_faiss.add(X_scaled)
index_faiss.nprobe = 10
t_build = time.perf_counter() - start

start = time.perf_counter()
idx_faiss = [index_faiss.search(v.reshape(1, -1), k)[1][0] for v in Xq]
t_query = time.perf_counter() - start
hasil.append(["FAISS IVF (nprobe=10)", t_build, t_query / n_query * 1000, recall_at_k(idx_faiss, idx_exact, k)])
print(f"FAISS IVF selesai: build {t_build:.3f} s, query {t_query:.3f} s")

FAISS IVF selesai: build 0.552 s, query 1.301 s


In [21]:
# -------------------------------
# Tabel perbandingan
# -------------------------------
tabel = pd.DataFrame(hasil, columns=["Metode", "Build (s)", "Query (ms/query)", "Recall@10"]).round(4)
print("\n", tabel.to_string(index=False))


                Metode  Build (s)  Query (ms/query)  Recall@10
     Exact NN (brute)     0.1112            5.3337     1.0000
      Annoy (10 tree)    20.7034            0.0530     0.8384
  HNSW (M=16, ef=200)   226.0822            0.2503     0.9967
FAISS IVF (nprobe=10)     0.5524            1.3005     0.9982


In [22]:
# -------------------------------
# Top-5 neighbors dari query pertama
# -------------------------------
print(f"\nTop-5 neighbors untuk lagu index {q_idx[0]}:")
print("Exact NN:", idx_exact[0][:5])
print("Annoy:   ", idx_annoy[0][:5])
print("HNSW:    ", idx_hnsw[0][:5])
print("FAISS:   ", idx_faiss[0][:5])


Top-5 neighbors untuk lagu index 287796:
Exact NN: [287796 275141 500828 610336 701566]
Annoy:    [287796, 275141, 500828, 610336, 701566]
HNSW:     [287796 275141 500828 610336 701566]
FAISS:    [287796 275141 500828 610336 701566]


## **Analisis Hasil Praktikum 6**

### 1. Gambaran percobaan
Dataset Spotify berisi **955.320 lagu** dengan 9 fitur audio (danceability, energy, loudness, speechiness, acousticness, instrumentalness, liveness, valence, tempo) yang sudah distandarisasi. Pencarian dilakukan untuk 1.000 lagu acak dengan k = 10 tetangga terdekat. Exact NN (brute force) dijadikan acuan kebenaran untuk menghitung Recall@10 pada tiga metode ANN.

### 2. Hasil perbandingan

| Metode | Build (s) | Query (ms/query) | Recall@10 | Percepatan query vs Exact |
|---|---|---|---|---|
| Exact NN (brute) | 0,11 | 5,334 | 1,0000 | 1x |
| Annoy (10 tree) | 20,70 | 0,053 | 0,8384 | sekitar 100x |
| HNSW (M=16, ef=200) | 226,08 | 0,250 | 0,9967 | sekitar 21x |
| FAISS IVF (nprobe=10) | 0,55 | 1,301 | 0,9982 | sekitar 4x |

### 3. Analisis per metode
- **Exact NN** selalu menghasilkan jawaban yang benar, tetapi harus menghitung jarak ke seluruh 955 ribu lagu untuk setiap query, sehingga waktunya paling lama (5,3 ms/query). Waktu build-nya hampir nol karena tidak ada struktur index yang dibuat.
- **Annoy** adalah yang tercepat saat query (0,053 ms, sekitar 100 kali lebih cepat dari Exact NN) dan build-nya moderat (20,7 detik). Kelemahannya, recall-nya paling rendah (0,838), artinya sekitar 16% tetangga yang ditemukan tidak sama dengan hasil exact.
- **HNSW** memberi recall sangat tinggi (0,997) dengan query yang cepat (0,25 ms), tetapi build-nya paling mahal, yaitu 226 detik (sekitar 3,8 menit), jauh lebih lama dari metode lain.
- **FAISS IVF** memberi recall tertinggi (0,998) dengan build tercepat di antara metode ANN (0,55 detik). Query-nya paling lambat di antara ANN (1,3 ms), tetapi masih sekitar 4 kali lebih cepat dari Exact NN.

### 4. Trade-off yang terlihat
Tidak ada metode yang unggul di semua aspek, karena setiap metode menukar tiga hal: **kecepatan query, akurasi (recall), dan biaya build**.
- Kecepatan query: Annoy > HNSW > FAISS IVF > Exact.
- Akurasi: FAISS IVF dan HNSW (hampir sempurna) > Annoy.
- Biaya build: FAISS IVF (paling murah) < Annoy < HNSW (paling mahal).

Biaya build baru terbayar jika query-nya banyak. Dengan selisih waktu per query terhadap Exact NN, perkiraan titik impasnya sekitar **110 query** untuk FAISS IVF, sekitar **3.900 query** untuk Annoy, dan sekitar **44.400 query** untuk HNSW. Artinya HNSW baru sepadan pada sistem rekomendasi yang melayani sangat banyak query dengan index yang jarang dibangun ulang.

### 5. Kesesuaian hasil pada contoh query
Pada lagu dengan index 287796, kelima tetangga teratas dari Exact NN, Annoy, HNSW, dan FAISS **identik** ([287796, 275141, 500828, 610336, 701566]). Tetangga pertama adalah lagu itu sendiri (jarak 0) karena query diambil dari dataset. Ini sejalan dengan recall yang tinggi: untuk sebagian besar query, tetangga paling dekat ditemukan dengan benar, dan perbedaan hasil baru muncul di posisi yang lebih jauh dalam daftar 10 tetangga.

### 6. Catatan dan keterbatasan
- Data hanya 9 dimensi dan memiliki struktur (lagu sejenis berkelompok), sehingga ANN bisa mencapai recall tinggi. Pada data acak berdimensi tinggi, recall umumnya jauh lebih rendah.
- Parameter belum disetel. Recall Annoy bisa dinaikkan dengan menambah jumlah tree atau memperbesar `search_k`, tetapi waktu query dan build ikut naik. Hal serupa berlaku untuk `nprobe` (FAISS) dan `ef` (HNSW).
- Waktu diukur satu kali di Colab, jadi bisa bervariasi antar run. Exact NN memproses 1.000 query sekaligus (batch), sedangkan metode ANN diuji satu query per panggilan, sehingga percepatan yang tertulis di atas cenderung konservatif untuk Exact NN.
- Contoh data awal memperlihatkan baris dengan banyak nilai 0 (misalnya tempo dan danceability 0), yang mungkin berasal dari data kosong yang diisi 0. Hal ini bisa memengaruhi jarak antar lagu, dan layak dicek kalau analisis dilanjutkan.

### 7. Kesimpulan
Untuk dataset ini, **FAISS IVF** memberi keseimbangan terbaik (recall 0,998, build hanya 0,55 detik, query 4 kali lebih cepat dari exact). **Annoy** cocok jika kecepatan query adalah prioritas utama dan recall sekitar 0,84 masih dapat diterima. **HNSW** cocok jika akurasi tinggi dan query cepat dibutuhkan pada skala besar, dan biaya build 226 detik bisa diterima karena index dibangun sekali lalu dipakai berulang. Pemilihan akhirnya bergantung pada kebutuhan sistem: seberapa sering data berubah, berapa banyak query, dan seberapa besar toleransi terhadap hasil yang kurang tepat.